In [405]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
import pickle

In [406]:
df = pd.read_csv("sensor.csv")

df = df.sort_values("time_ms").reset_index(drop=True)

In [407]:
# keep only valid readings
df = df[df["valid"] == 1].copy()

In [408]:
# remove rows before baseline is learned
df = df[df["baseline_cm"] > 0].copy()

In [409]:
# remove impossible distances if any slipped in
df = df[(df["dist_f_cm"] > 20) & (df["dist_f_cm"] <= 600)].copy()

In [410]:
print("After cleaning:", df.shape)

After cleaning: (23014, 15)


In [411]:
# =========================
# 4. FEATURE ENGINEERING
# =========================

In [412]:
# main distance difference from baseline
df["delta"] = df["dist_f_cm"] - df["baseline_cm"]

In [413]:
df.head()

,Unnamed: 0,session_id,run_type,scenario,time_ms,sensor_id,echo_us,valid,dist_cm,dist_f_cm,baseline_cm,enter_thr_cm,exit_thr_cm,danger,event,delta
37,112,1,C_front_far_deep,safe,31798,1,3646,1,269.8,267.5,268.3,120.0,140.0,0.0,0,-0.8
38,113,1,C_front_far_deep,safe,32080,1,3629,1,268.5,267.8,268.3,208.3,228.3,0.0,0,-0.5
39,114,1,C_front_far_deep,safe,32363,1,3657,1,270.6,268.5,268.3,208.3,228.3,0.0,0,0.2
40,115,1,C_front_far_deep,safe,32632,1,3659,1,270.8,269.1,268.3,208.3,228.3,0.0,0,0.8
41,116,1,C_front_far_deep,safe,32902,1,3627,1,268.4,268.9,268.3,208.3,228.3,0.0,0,0.6


In [414]:
# first row contains a hardcoded enter and exit thresholds. must remove them
#df = df[df["delta"] != 0].copy()

In [415]:
#df.head()

In [416]:
# main distance difference from baseline
df["delta"] = df["dist_f_cm"] - df["baseline_cm"]

In [417]:
# normalized delta (helps when baseline changes across sessions/sensors)
df["delta_norm"] = df["delta"] / df["baseline_cm"]

In [418]:
# distance to thresholds
df["dist_to_enter"] = df["dist_f_cm"] - df["enter_thr_cm"]
df["dist_to_exit"] = df["dist_f_cm"] - df["exit_thr_cm"]

In [419]:
group_cols = []
if "session_id" in df.columns:
    group_cols.append("session_id")

In [420]:
df["velocity"] = df.groupby(group_cols)["dist_f_cm"].diff()
df["acceleration"] = df.groupby(group_cols)["velocity"].diff()

In [421]:
df["time_gap"] =df.groupby(group_cols)["time_ms"].diff()

In [422]:
df.head()

,Unnamed: 0,session_id,run_type,scenario,time_ms,sensor_id,echo_us,valid,dist_cm,dist_f_cm,...,exit_thr_cm,danger,event,delta,delta_norm,dist_to_enter,dist_to_exit,velocity,acceleration,time_gap
37,112,1,C_front_far_deep,safe,31798,1,3646,1,269.8,267.5,...,140.0,0.0,0,-0.8,-0.002982,147.5,127.5,NaN,NaN,NaN
38,113,1,C_front_far_deep,safe,32080,1,3629,1,268.5,267.8,...,228.3,0.0,0,-0.5,-0.001864,59.5,39.5,0.3,NaN,282.0
39,114,1,C_front_far_deep,safe,32363,1,3657,1,270.6,268.5,...,228.3,0.0,0,0.2,0.000745,60.2,40.2,0.7,0.4,283.0
40,115,1,C_front_far_deep,safe,32632,1,3659,1,270.8,269.1,...,228.3,0.0,0,0.8,0.002982,60.8,40.8,0.6,-0.1,269.0
41,116,1,C_front_far_deep,safe,32902,1,3627,1,268.4,268.9,...,228.3,0.0,0,0.6,0.002236,60.6,40.6,-0.2,-0.8,270.0


In [423]:
print("\nAfter feature engineering:")
print(df.head())
df.shape


After feature engineering:
    Unnamed: 0  session_id          run_type scenario  time_ms  sensor_id  \
37         112           1  C_front_far_deep     safe    31798          1   
38         113           1  C_front_far_deep     safe    32080          1   
39         114           1  C_front_far_deep     safe    32363          1   
40         115           1  C_front_far_deep     safe    32632          1   
41         116           1  C_front_far_deep     safe    32902          1   

    echo_us  valid  dist_cm  dist_f_cm  ...  exit_thr_cm  danger  event  \
37     3646      1    269.8      267.5  ...        140.0     0.0      0   
38     3629      1    268.5      267.8  ...        228.3     0.0      0   
39     3657      1    270.6      268.5  ...        228.3     0.0      0   
40     3659      1    270.8      269.1  ...        228.3     0.0      0   
41     3627      1    268.4      268.9  ...        228.3     0.0      0   

    delta  delta_norm  dist_to_enter  dist_to_exit  veloci

(23014, 22)

In [424]:
# feature_fill_cols = ["velocity", "acceleration", "time_gap"]
# for col in feature_fill_cols:
#     if col in df.columns:
#         df[col] = df[col].fillna(0)

In [425]:
print("\nAfter NaN handling:", df.shape)


After NaN handling: (23014, 22)


In [426]:
PRED_HORIZON = 3

df["future_danger"] = df["danger"].shift(-PRED_HORIZON)
df = df.dropna().reset_index(drop=True)

In [427]:
df.shape

(22831, 23)

In [428]:
n = len(df)

train_end = int(0.7 * n)
val_end = int(0.85 * n)

train_df = df.iloc[:train_end].reset_index(drop=True)
val_df   = df.iloc[train_end:val_end].reset_index(drop=True)
test_df  = df.iloc[val_end:].reset_index(drop=True)

In [429]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)


(15981, 23)
(3425, 23)
(3425, 23)


In [430]:
feature_cols = [
    "time_gap",
    "dist_f_cm",
    "delta",
    "delta_norm",
    "velocity",
    "acceleration"
]
label_col = "future_danger"


In [431]:
# if "state" not in df.columns:
#     df["state"] = (df["danger"] > 0).astype(int)
#     label_col = "state"

In [432]:
print("\nUsing features:", feature_cols)
print("Using label:", label_col)


Using features: ['time_gap', 'dist_f_cm', 'delta', 'delta_norm', 'velocity', 'acceleration']
Using label: future_danger


In [433]:
df.head(1000)

,Unnamed: 0,session_id,run_type,scenario,time_ms,sensor_id,echo_us,valid,dist_cm,dist_f_cm,...,danger,event,delta,delta_norm,dist_to_enter,dist_to_exit,velocity,acceleration,time_gap,future_danger
0,114,1,C_front_far_deep,safe,32363,1,3657,1,270.6,268.5,...,0.0,0,0.2,0.000745,60.2,40.2,0.7,0.4,283.0,0.000
1,115,1,C_front_far_deep,safe,32632,1,3659,1,270.8,269.1,...,0.0,0,0.8,0.002982,60.8,40.8,0.6,-0.1,269.0,0.000
2,116,1,C_front_far_deep,safe,32902,1,3627,1,268.4,268.9,...,0.0,0,0.6,0.002236,60.6,40.6,-0.2,-0.8,270.0,0.000
3,117,1,C_front_far_deep,safe,33190,1,3656,1,270.5,269.3,...,0.0,0,1.0,0.003727,61.0,41.0,0.4,0.6,288.0,0.000
4,118,1,C_front_far_deep,safe,33476,1,3667,1,271.4,269.8,...,0.0,0,1.5,0.005591,61.5,41.5,0.5,0.1,286.0,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,4716,4,C_front_far_deep,approach_danger,1327503,1,2412,1,178.5,174.1,...,1.0,0,-91.8,-0.345243,-31.8,-51.8,1.5,2.5,280.0,1.000
996,4717,4,C_front_far_deep,approach_danger,1327786,1,2569,1,190.1,178.1,...,1.0,0,-87.8,-0.330199,-27.8,-47.8,4.0,2.5,283.0,1.000
997,4718,4,C_front_far_deep,approach_danger,1328061,1,2664,1,197.1,182.9,...,1.0,0,-83.0,-0.312147,-23.0,-43.0,4.8,0.8,275.0,1.000
998,4719,4,C_front_far_deep,approach_danger,1328348,1,2795,1,206.8,188.8,...,1.0,0,-77.1,-0.289959,-17.1,-37.1,5.9,1.1,287.0,0.939


In [434]:
# # =========================
# # 7. NORMALIZE FEATURES
# # =========================
# # z-score normalization
# feature_means = df[feature_cols].mean()
# feature_stds = df[feature_cols].std().replace(0, 1)

# df_norm = df.copy()
# df_norm[feature_cols] = (df[feature_cols] - feature_means) / feature_stds

# print("\nNormalized sample:")
# print(df_norm[feature_cols].head())

In [435]:
# df_norm.head()

In [436]:
# print(group_cols)
# df_norm.shape

In [437]:
# n = len(df_norm)

# train_end = int(0.7 * n)
# val_end = int(0.85 * n)

# train_df = df_norm.iloc[:train_end]
# val_df   = df_norm.iloc[train_end:val_end]
# test_df  = df_norm.iloc[val_end:]

# print("Train shape:", train_df.shape)
# print("Val shape:", val_df.shape)
# print("Test shape:", test_df.shape)

In [438]:
# print(feature_cols)
# print(label_col)
# train_df.head(1000)

In [439]:
def create_windows(df, window_size, step_size):

    X = []
    y = []

    for start in range(0, len(df) - window_size, step_size):

        end = start + window_size

        window_features = df.iloc[start:end][feature_cols].values
        window_label = df.iloc[end - 1][label_col]

        X.append(window_features)
        y.append(window_label)

    return np.array(X), np.array(y)

In [440]:
WINDOW_SIZE = 10
STEP_SIZE = 3

X_train, y_train = create_windows(train_df, WINDOW_SIZE, STEP_SIZE)
X_val, y_val     = create_windows(val_df, WINDOW_SIZE, STEP_SIZE)
X_test, y_test   = create_windows(test_df, WINDOW_SIZE, STEP_SIZE)

In [402]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(5324, 10, 6)
(1139, 10, 6)
(1139, 10, 6)


In [404]:
#unique value counts in y_train
# Shows each unique value and its frequency
y_train.unique()


AttributeError: 'numpy.ndarray' object has no attribute 'unique'

In [ ]:
# X = np.array(X, dtype=np.float32)
# y = np.array(y, dtype=np.int64)

# print("\nWindowed dataset shapes:")
# print("X shape:", X.shape)   # (samples, time_steps, features)
# print("y shape:", y.shape)


Windowed dataset shapes:
X shape: (21754, 10, 6)
y shape: (21754,)


In [ ]:
# # =========================
# # 9. SAVE OUTPUTS
# # =========================
# # cleaned flat table
# df.to_csv("sensor_preprocessed_flat.csv", index=False)

# # normalized flat table
# df_norm.to_csv("sensor_preprocessed_normalized.csv", index=False)

# # SNN-ready arrays
# np.save("X_snn.npy", X)
# np.save("y_snn.npy", y)

# print("\nSaved files:")
# print("sensor_preprocessed_flat.csv")
# print("sensor_preprocessed_normalized.csv")
# print("X_snn.npy")
# print("y_snn.npy")


Saved files:
sensor_preprocessed_flat.csv
sensor_preprocessed_normalized.csv
X_snn.npy
y_snn.npy


In [ ]:
# print(X[50].shape)
# print(X[50])
# print("Label:", y[155])

(10, 6)
[[-0.20804058  1.278805    0.63467866  0.63229614 -0.00672242  0.1830439 ]
 [-0.20414636  1.2696761   0.6248009   0.62329596 -0.04890644 -0.0509632 ]
 [-0.20268601  1.2392464   0.5918753   0.5932954  -0.19655049 -0.16796674]
 [-0.20268601  1.2118596   0.56224227  0.5662949  -0.17545849  0.01923893]
 [-0.20171246  1.1662151   0.5128538   0.52129406 -0.30201054 -0.14456603]
 [-0.20414636  1.1449143   0.48980588  0.5002937  -0.13327447  0.1830439 ]
 [-0.2077972   1.1296993   0.47334304  0.48529342 -0.09109046  0.04263964]
 [-0.20438974  1.1418712   0.48651332  0.49729362  0.09873761  0.2064446 ]
 [-0.20438974  1.1601291   0.5062687   0.51529396  0.14092162  0.04263964]
 [-0.20414636  1.1510001   0.496391    0.5062938  -0.04890644 -0.21476817]]
Label: 0


In [ ]:
unique, counts = np.unique(y_train, return_counts=True)


{np.float64(0.0): np.int64(4382), np.float64(0.741): np.int64(1), np.float64(0.753): np.int64(1), np.float64(0.758): np.int64(1), np.float64(0.761): np.int64(1), np.float64(0.765): np.int64(1), np.float64(0.768): np.int64(2), np.float64(0.769): np.int64(2), np.float64(0.77): np.int64(1), np.float64(0.772): np.int64(1), np.float64(0.773): np.int64(2), np.float64(0.78): np.int64(1), np.float64(0.781): np.int64(1), np.float64(0.783): np.int64(1), np.float64(0.784): np.int64(1), np.float64(0.786): np.int64(1), np.float64(0.79): np.int64(1), np.float64(0.792): np.int64(2), np.float64(0.795): np.int64(2), np.float64(0.797): np.int64(2), np.float64(0.798): np.int64(2), np.float64(0.799): np.int64(1), np.float64(0.8): np.int64(1), np.float64(0.802): np.int64(1), np.float64(0.806): np.int64(1), np.float64(0.808): np.int64(1), np.float64(0.809): np.int64(2), np.float64(0.81): np.int64(1), np.float64(0.812): np.int64(1), np.float64(0.813): np.int64(1), np.float64(0.815): np.int64(3), np.float64(0

In [ ]:
# # total number of windows
# n = len(X)

# # 70% train, 15% validation, 15% test
# train_end = int(0.70 * n)
# val_end = int(0.85 * n)

# X_train = X[:train_end]
# y_train = y[:train_end]

# X_val = X[train_end:val_end]
# y_val = y[train_end:val_end]

# X_test = X[val_end:]
# y_test = y[val_end:]

# print("Train:", X_train.shape, y_train.shape)
# print("Val  :", X_val.shape, y_val.shape)
# print("Test :", X_test.shape, y_test.shape)

Train: (15227, 10, 6) (15227,)
Val  : (3263, 10, 6) (3263,)
Test : (3264, 10, 6) (3264,)


In [241]:
def label_counts(name, labels):
    unique, counts = np.unique(labels, return_counts=True)
    print(name, dict(zip(unique, counts)))

label_counts("Train", y_train)
label_counts("Val", y_val)
label_counts("Test", y_test)

Train {np.float64(0.0): np.int64(4382), np.float64(0.741): np.int64(1), np.float64(0.753): np.int64(1), np.float64(0.758): np.int64(1), np.float64(0.761): np.int64(1), np.float64(0.765): np.int64(1), np.float64(0.768): np.int64(2), np.float64(0.769): np.int64(2), np.float64(0.77): np.int64(1), np.float64(0.772): np.int64(1), np.float64(0.773): np.int64(2), np.float64(0.78): np.int64(1), np.float64(0.781): np.int64(1), np.float64(0.783): np.int64(1), np.float64(0.784): np.int64(1), np.float64(0.786): np.int64(1), np.float64(0.79): np.int64(1), np.float64(0.792): np.int64(2), np.float64(0.795): np.int64(2), np.float64(0.797): np.int64(2), np.float64(0.798): np.int64(2), np.float64(0.799): np.int64(1), np.float64(0.8): np.int64(1), np.float64(0.802): np.int64(1), np.float64(0.806): np.int64(1), np.float64(0.808): np.int64(1), np.float64(0.809): np.int64(2), np.float64(0.81): np.int64(1), np.float64(0.812): np.int64(1), np.float64(0.813): np.int64(1), np.float64(0.815): np.int64(3), np.flo

In [110]:
# Save original shapes
n_train, t_train, f_train = X_train.shape
n_val, t_val, f_val = X_val.shape
n_test, t_test, f_test = X_test.shape

In [111]:
# Reshape 3D -> 2D
X_train_2d = X_train.reshape(-1, f_train)
X_val_2d = X_val.reshape(-1, f_val)
X_test_2d = X_test.reshape(-1, f_test)

In [112]:
from sklearn.preprocessing import StandardScaler
# Fit scaler ONLY on training data
scaler = StandardScaler()
X_train_scaled_2d = scaler.fit_transform(X_train_2d)

In [113]:
# Apply same scaler to val/test
X_val_scaled_2d = scaler.transform(X_val_2d)
X_test_scaled_2d = scaler.transform(X_test_2d)

In [114]:
# Reshape back 2D -> 3D
X_train_scaled = X_train_scaled_2d.reshape(n_train, t_train, f_train)
X_val_scaled = X_val_scaled_2d.reshape(n_val, t_val, f_val)
X_test_scaled = X_test_scaled_2d.reshape(n_test, t_test, f_test)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled val shape  :", X_val_scaled.shape)
print("Scaled test shape :", X_test_scaled.shape)

Scaled train shape: (5366, 10, 6)
Scaled val shape  : (1147, 10, 6)
Scaled test shape : (1147, 10, 6)


In [115]:
# X_train_scaled = X_train
# X_val_scaled = X_val
# X_test_scaled = X_test

In [116]:
print("Before scaling:")
print(X_train[1])

print("\nAfter scaling:")
print(X_train_scaled[1])

Before scaling:
[[-0.2077972   1.29401987  0.65114145  0.64729641  0.14092163 -0.02756249]
 [-0.20755381  1.28793392  0.64455632  0.6412963  -0.02781443 -0.19136745]
 [-0.2031728   1.30010581  0.65772658  0.65329652  0.09873761  0.13624248]
 [-0.20365957  1.31532067  0.67418939  0.6682968   0.11982962  0.01923893]
 [-0.20438974  1.33357851  0.69394477  0.68629713  0.14092163  0.01923893]
 [-0.20268602  1.35487931  0.71699271  0.70729752  0.16201364  0.01923893]
 [-0.20268602  1.388352    0.7532109   0.74029813  0.24638167  0.08944106]
 [-0.20463313  1.39748092  0.76308859  0.7492983   0.07764561 -0.19136745]
 [-0.20390296  1.39748092  0.76308859  0.7492983   0.01436958 -0.07436391]
 [-0.20438974  1.39443795  0.75979603  0.74629824 -0.00672242 -0.02756249]]

After scaling:
[[-0.20837097  1.26005308  0.63968721  0.63624478  0.14208282 -0.02405381]
 [-0.20812817  1.2539676   0.63314771  0.63024846 -0.02577082 -0.18296285]
 [-0.20375785  1.26613855  0.64622672  0.6422411   0.10011941  0.13

In [139]:
!pip install torch torchvision torchaudio

In [117]:
import torch

In [118]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

print(X_train_t.shape, y_train_t.shape)

torch.Size([5366, 10, 6]) torch.Size([5366])


In [119]:
#Create DataLoaders
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64

train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [120]:
#Handle Class Imbalance
#penalize mistakes on danger more strongly

In [121]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float32)

print(class_weights)

tensor([0.6056, 2.8665])


In [122]:
#Loss function with class weights
import torch.nn as nn

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [146]:
!pip install snntorch

In [123]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import snntorch as snn
from snntorch import surrogate
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [124]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [125]:
class_weights = class_weights.to(device)

In [126]:
#the SNN model
class SimpleSNN(nn.Module):
    def __init__(self, input_size=6, hidden_size=32, output_size=2, beta=0.9):
        super().__init__()

        # Fully connected layers
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

        # Surrogate gradient for backpropagation through spikes
        spike_grad = surrogate.fast_sigmoid()

        # LIF neurons
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x):
        batch_size = x.size(0)
        time_steps = x.size(1)

        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()

        mem3_rec = []

        for t in range(time_steps):
            cur1 = self.fc1(x[:, t, :])
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)

            cur3 = self.fc3(spk2)
            spk3, mem3 = self.lif3(cur3, mem3)

            mem3_rec.append(mem3)

        mem3_rec = torch.stack(mem3_rec, dim=0)   # (time_steps, batch, output_size)
        return mem3_rec



In [127]:
#Create the model
model = SimpleSNN(input_size=6, hidden_size=32, output_size=2, beta=0.9).to(device)
print(model)

SimpleSNN(
  (fc1): Linear(in_features=6, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=2, bias=True)
  (lif1): Leaky()
  (lif2): Leaky()
  (lif3): Leaky()
)


In [128]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [129]:
#Training loop
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        mem_rec = model(X_batch)          # shape: (time_steps, batch, 2)
        output = mem_rec[-1]              # last time step -> (batch, 2)

        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(output, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc

In [130]:
# Validation loop
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            mem_rec = model(X_batch)
            output = mem_rec[-1]

            loss = criterion(output, y_batch)
            running_loss += loss.item() * X_batch.size(0)

            preds = torch.argmax(output, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc, all_labels, all_preds

In [131]:
num_epochs = 20

train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch [1/20] Train Loss: 0.3992 | Train Acc: 0.6759 | Val Loss: 0.1220 | Val Acc: 0.9712
Epoch [2/20] Train Loss: 0.0871 | Train Acc: 0.9683 | Val Loss: 0.0899 | Val Acc: 0.9721
Epoch [3/20] Train Loss: 0.0704 | Train Acc: 0.9778 | Val Loss: 0.0865 | Val Acc: 0.9765
Epoch [4/20] Train Loss: 0.0702 | Train Acc: 0.9745 | Val Loss: 0.0816 | Val Acc: 0.9730
Epoch [5/20] Train Loss: 0.0690 | Train Acc: 0.9741 | Val Loss: 0.0970 | Val Acc: 0.9765
Epoch [6/20] Train Loss: 0.0700 | Train Acc: 0.9765 | Val Loss: 0.0793 | Val Acc: 0.9791
Epoch [7/20] Train Loss: 0.0689 | Train Acc: 0.9761 | Val Loss: 0.0813 | Val Acc: 0.9773
Epoch [8/20] Train Loss: 0.0670 | Train Acc: 0.9747 | Val Loss: 0.0863 | Val Acc: 0.9747
Epoch [9/20] Train Loss: 0.0679 | Train Acc: 0.9734 | Val Loss: 0.0865 | Val Acc: 0.9721
Epoch [10/20] Train Loss: 0.0670 | Train Acc: 0.9760 | Val Loss: 0.0934 | Val Acc: 0.9765
Epoch [11/20] Train Loss: 0.0678 | Train Acc: 0.9767 | Val Loss: 0.0771 | Val Acc: 0.9738
Epoch [12/20] Train

In [132]:
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc : {test_acc:.4f}")

Test Loss: 0.0834
Test Acc : 0.9747


In [133]:
print(feature_cols)

['time_gap', 'dist_f_cm', 'delta', 'delta_norm', 'velocity', 'acceleration']


In [134]:
y_train_random = np.random.permutation(y_train)

print("Original label distribution:", np.mean(y_train))
print("Randomized label distribution:", np.mean(y_train_random))

Original label distribution: 0.17443160641073424
Randomized label distribution: 0.17443160641073424
